# Virginia Regional Economic Intelligence

## 01 — Data Acquisition

This notebook builds the raw data layer for an applied regional economic analysis of Virginia.

### Objectives
- Programmatically acquire public labor-market data.
- Restrict the analysis to Virginia localities.
- Preserve raw source data for reproducibility.
- Perform initial structural checks before downstream cleaning and analysis.

### Primary Data Source
**U.S. Bureau of Labor Statistics — Quarterly Census of Employment and Wages (QCEW)**

The QCEW provides employment, establishment, and wage data by geography and industry and will serve as the primary labor-market dataset for this project.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import requests
import io
import zipfile

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
# Project paths

PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

OUTPUT_FIGURES = PROJECT_ROOT / "outputs" / "figures"
OUTPUT_TABLES = PROJECT_ROOT / "outputs" / "tables"

for directory in [
    RAW_DIR,
    INTERIM_DIR,
    PROCESSED_DIR,
    OUTPUT_FIGURES,
    OUTPUT_TABLES
]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DIR}")

Project root: C:\Users\Chris\Documents\virginia-regional-economic-intelligence
Raw data directory: C:\Users\Chris\Documents\virginia-regional-economic-intelligence\data\raw


## 1. BLS QCEW Data Acquisition

The Quarterly Census of Employment and Wages (QCEW) provides employment and wage data by geography, industry, and ownership.

For this project, annual-average QCEW data from 2019 through 2025 will be collected programmatically. Using 2019 as the starting point provides a pre-pandemic baseline while retaining enough recent history to evaluate structural changes in Virginia's regional economies.

Raw source files are preserved separately from processed data to support reproducibility and data lineage.

In [3]:
# Analysis period

START_YEAR = 2019
END_YEAR = 2025

YEARS = list(range(START_YEAR, END_YEAR + 1))

YEARS

[2019, 2020, 2021, 2022, 2023, 2024, 2025]

In [4]:
# BLS QCEW annual single-file download URLs

BLS_BASE_URL = "https://data.bls.gov/cew/data/files"

qcew_urls = {
    year: f"{BLS_BASE_URL}/{year}/csv/{year}_annual_singlefile.zip"
    for year in YEARS
}

qcew_urls

{2019: 'https://data.bls.gov/cew/data/files/2019/csv/2019_annual_singlefile.zip',
 2020: 'https://data.bls.gov/cew/data/files/2020/csv/2020_annual_singlefile.zip',
 2021: 'https://data.bls.gov/cew/data/files/2021/csv/2021_annual_singlefile.zip',
 2022: 'https://data.bls.gov/cew/data/files/2022/csv/2022_annual_singlefile.zip',
 2023: 'https://data.bls.gov/cew/data/files/2023/csv/2023_annual_singlefile.zip',
 2024: 'https://data.bls.gov/cew/data/files/2024/csv/2024_annual_singlefile.zip',
 2025: 'https://data.bls.gov/cew/data/files/2025/csv/2025_annual_singlefile.zip'}

In [5]:
# Test the most recent annual file before running the full download

test_year = 2025
test_url = qcew_urls[test_year]

print(test_url)

response = requests.get(test_url, timeout=60)

print(f"HTTP status: {response.status_code}")
print(f"Downloaded size: {len(response.content) / 1_000_000:.2f} MB")

https://data.bls.gov/cew/data/files/2025/csv/2025_annual_singlefile.zip
HTTP status: 200
Downloaded size: 62.86 MB


## 2. Download and Preserve Raw QCEW Files

To support reproducibility, each annual QCEW archive is downloaded directly from the Bureau of Labor Statistics and stored in the project's raw-data directory.

The raw files are not modified at this stage. Preserving the original source files creates a clear data lineage and allows the analysis to be reproduced or audited later.

Before downloading, the workflow checks whether a file already exists locally. This avoids unnecessary repeated downloads and reduces dependence on the external data source.

In [6]:
# Download annual BLS QCEW archives
# ---------------------------------
# Each ZIP file is saved exactly as provided by BLS.
# Existing files are skipped so the notebook can be rerun safely.

def download_qcew_file(year, url, output_dir):
    """
    Download one annual BLS QCEW ZIP archive.

    Parameters
    ----------
    year : int
        Calendar year of the QCEW annual file.

    url : str
        BLS download URL.

    output_dir : pathlib.Path
        Directory where the raw ZIP file will be stored.

    Returns
    -------
    pathlib.Path
        Local path to the downloaded or existing file.
    """

    output_path = output_dir / f"{year}_annual_singlefile.zip"

    # Avoid downloading the same raw source file more than once.
    if output_path.exists():
        print(f"{year}: file already exists — skipping download.")
        return output_path

    print(f"{year}: downloading...")

    response = requests.get(url, timeout=120)

    # Raise an explicit error if the BLS request was unsuccessful.
    response.raise_for_status()

    # Write the original binary ZIP archive to the raw-data directory.
    output_path.write_bytes(response.content)

    file_size_mb = output_path.stat().st_size / 1_000_000

    print(f"{year}: complete ({file_size_mb:.2f} MB)")

    return output_path

In [7]:
# Download all annual QCEW files in the study period.

qcew_raw_files = {}

for year, url in qcew_urls.items():
    qcew_raw_files[year] = download_qcew_file(
        year=year,
        url=url,
        output_dir=RAW_DIR
    )

2019: file already exists — skipping download.
2020: file already exists — skipping download.
2021: file already exists — skipping download.
2022: file already exists — skipping download.
2023: file already exists — skipping download.
2024: file already exists — skipping download.
2025: file already exists — skipping download.


In [8]:
# Verify that every expected annual archive is present locally.

download_summary = pd.DataFrame(
    {
        "year": qcew_raw_files.keys(),
        "file_path": [str(path) for path in qcew_raw_files.values()],
        "file_size_mb": [
            path.stat().st_size / 1_000_000
            for path in qcew_raw_files.values()
        ]
    }
)

download_summary

,year,file_path,file_size_mb
0,2019,C:\Users\Chris\Documents\virginia-regional-eco...,78.411719
1,2020,C:\Users\Chris\Documents\virginia-regional-eco...,77.441967
2,2021,C:\Users\Chris\Documents\virginia-regional-eco...,79.217997
3,2022,C:\Users\Chris\Documents\virginia-regional-eco...,77.024919
4,2023,C:\Users\Chris\Documents\virginia-regional-eco...,82.932544
5,2024,C:\Users\Chris\Documents\virginia-regional-eco...,74.697761
6,2025,C:\Users\Chris\Documents\virginia-regional-eco...,62.856340


### Acquisition Check

The annual source archives were successfully retrieved and preserved in the raw-data layer.

The next step is to inspect the internal structure of the QCEW files before deciding which geographic and industry records are appropriate for the regional economic analysis.

## 3. Inspect the QCEW File Structure

Before filtering or transforming the QCEW data, the structure of the source files is examined to understand how BLS represents geography, industry, ownership, employment, and wages.

This step is important because the annual singlefile contains multiple levels of geographic and industry aggregation. A valid regional analysis requires deliberately selecting the appropriate records rather than assuming that every row represents a county-industry observation.

The 2025 file is inspected first as a representative example before applying any processing logic across the full study period.

In [9]:
# Inspect the contents of the 2025 BLS ZIP archive
# ------------------------------------------------
# We examine the archive structure first rather than extracting files
# unnecessarily to disk.

sample_year = 2025
sample_zip_path = qcew_raw_files[sample_year]

with zipfile.ZipFile(sample_zip_path, "r") as z:
    zip_contents = z.namelist()

zip_contents

['2025.annual.singlefile.csv']

In [10]:
# Display basic archive metadata.

with zipfile.ZipFile(sample_zip_path, "r") as z:
    archive_info = pd.DataFrame(
        {
            "file_name": [info.filename for info in z.infolist()],
            "compressed_mb": [
                info.compress_size / 1_000_000
                for info in z.infolist()
            ],
            "uncompressed_mb": [
                info.file_size / 1_000_000
                for info in z.infolist()
            ]
        }
    )

archive_info

,file_name,compressed_mb,uncompressed_mb
0,2025.annual.singlefile.csv,62.856154,439.500013


### Initial Schema Inspection

Rather than loading the entire national dataset immediately, a small sample of records is read first.

This allows the field names, data types, coding conventions, and aggregation structure to be reviewed with minimal memory usage. The results will determine the filtering strategy used to construct the Virginia analytical dataset.

In [11]:
# Read a small sample from the annual CSV.
# Only the first 10 rows are loaded at this stage so that the schema
# can be examined without loading the full national dataset into memory.

with zipfile.ZipFile(sample_zip_path, "r") as z:
    csv_name = z.namelist()[0]

    with z.open(csv_name) as csv_file:
        qcew_sample = pd.read_csv(
            csv_file,
            nrows=10,
            dtype=str
        )

qcew_sample

,area_fips,own_code,industry_code,agglvl_code,size_code,year,qtr,disclosure_code,annual_avg_estabs,annual_avg_emplvl,total_annual_wages,taxable_annual_wages,annual_contributions,annual_avg_wkly_wage,avg_annual_pay,lq_disclosure_code,lq_annual_avg_estabs,lq_annual_avg_emplvl,lq_total_annual_wages,lq_taxable_annual_wages,lq_annual_contributions,lq_annual_avg_wkly_wage,lq_avg_annual_pay,oty_disclosure_code,oty_annual_avg_estabs_chg,oty_annual_avg_estabs_pct_chg,oty_annual_avg_emplvl_chg,oty_annual_avg_emplvl_pct_chg,oty_total_annual_wages_chg,oty_total_annual_wages_pct_chg,oty_taxable_annual_wages_chg,oty_taxable_annual_wages_pct_chg,oty_annual_contributions_chg,oty_annual_contributions_pct_chg,oty_annual_avg_wkly_wage_chg,oty_annual_avg_wkly_wage_pct_chg,oty_avg_annual_pay_chg,oty_avg_annual_pay_pct_chg
0,01000,0,10,50,0,2025,A,NaN,164938,2119596,136657263309,17594046587,76878255,1240,64473,NaN,1.00,1.00,1.00,1.00,1.00,1.00,1.00,NaN,4051,2.5,13922,0.7,6012446239,4.6,39973460,0.2,-12470976,-14.0,47,3.9,2429,3.9
1,01000,1,10,51,0,2025,A,NaN,1270,57362,5947694382,0,0,1994,103687,NaN,1.55,1.45,1.74,0.00,0.00,1.20,1.20,NaN,1,0.1,-770,-1.3,91626227,1.6,0,0.0,0,0.0,57,2.9,2949,2.9
2,01000,1,101,52,0,2025,A,NaN,2,2,167889,0,0,1937,100733,NaN,0.91,0.00,0.00,0.00,0.00,1.18,1.18,NaN,1,100.0,1,100.0,109786,189.0,0,0.0,0,0.0,447,30.0,23262,30.0
3,01000,1,1013,53,0,2025,A,NaN,2,2,167889,0,0,1937,100733,NaN,1.73,0.00,0.00,0.00,0.00,1.18,1.18,NaN,1,100.0,1,100.0,109786,189.0,0,0.0,0,0.0,447,30.0,23262,30.0
4,01000,1,102,52,0,2025,A,NaN,1268,57360,5947526493,0,0,1994,103688,NaN,1.55,1.47,1.77,0.00,0.00,1.20,1.20,NaN,0,0.0,-771,-1.3,91516441,1.6,0,0.0,0,0.0,57,2.9,2950,2.9
5,01000,1,1021,53,0,2025,A,NaN,614,11798,923956429,0,0,1506,78317,NaN,1.49,1.32,1.69,0.00,0.00,1.28,1.28,NaN,-4,-0.6,-73,-0.6,9436247,1.0,0,0.0,0,0.0,25,1.7,1279,1.7
6,01000,1,1022,53,0,2025,A,NaN,2,11,427011,0,0,770,40032,NaN,0.98,0.13,0.06,0.00,0.00,0.48,0.48,NaN,0,0.0,1,10.0,46505,12.2,0,0.0,0,0.0,26,3.5,1336,3.5
7,01000,1,1023,53,0,2025,A,NaN,22,148,17096066,0,0,2214,115125,NaN,1.97,0.87,0.78,0.00,0.00,0.89,0.89,NaN,2,10.0,-7,-4.5,-46019,-0.3,0,0.0,0,0.0,92,4.3,4768,4.3
8,01000,1,1024,53,0,2025,A,NaN,43,1956,173678813,0,0,1707,88778,NaN,2.07,2.02,2.15,0.00,0.00,1.06,1.06,NaN,-1,-2.3,52,2.7,-556218,-0.3,0,0.0,0,0.0,-52,-3.0,-2712,-3.0
9,01000,1,1025,53,0,2025,A,NaN,43,7392,733363856,0,0,1908,99205,NaN,2.15,1.24,1.36,0.00,0.00,1.09,1.09,NaN,0,0.0,-268,-3.5,34208809,4.9,0,0.0,0,0.0,153,8.7,7936,8.7


In [12]:
# Review the available variables.

print(f"Number of columns: {qcew_sample.shape[1]}")

qcew_sample.columns.tolist()

Number of columns: 38


['area_fips',
 'own_code',
 'industry_code',
 'agglvl_code',
 'size_code',
 'year',
 'qtr',
 'disclosure_code',
 'annual_avg_estabs',
 'annual_avg_emplvl',
 'total_annual_wages',
 'taxable_annual_wages',
 'annual_contributions',
 'annual_avg_wkly_wage',
 'avg_annual_pay',
 'lq_disclosure_code',
 'lq_annual_avg_estabs',
 'lq_annual_avg_emplvl',
 'lq_total_annual_wages',
 'lq_taxable_annual_wages',
 'lq_annual_contributions',
 'lq_annual_avg_wkly_wage',
 'lq_avg_annual_pay',
 'oty_disclosure_code',
 'oty_annual_avg_estabs_chg',
 'oty_annual_avg_estabs_pct_chg',
 'oty_annual_avg_emplvl_chg',
 'oty_annual_avg_emplvl_pct_chg',
 'oty_total_annual_wages_chg',
 'oty_total_annual_wages_pct_chg',
 'oty_taxable_annual_wages_chg',
 'oty_taxable_annual_wages_pct_chg',
 'oty_annual_contributions_chg',
 'oty_annual_contributions_pct_chg',
 'oty_annual_avg_wkly_wage_chg',
 'oty_annual_avg_wkly_wage_pct_chg',
 'oty_avg_annual_pay_chg',
 'oty_avg_annual_pay_pct_chg']

In [13]:
# Inspect sample values and Python data types.

qcew_sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 38 columns):
 #   Column                            Non-Null Count  Dtype 
---  ------                            --------------  ----- 
 0   area_fips                         10 non-null     object
 1   own_code                          10 non-null     object
 2   industry_code                     10 non-null     object
 3   agglvl_code                       10 non-null     object
 4   size_code                         10 non-null     object
 5   year                              10 non-null     object
 6   qtr                               10 non-null     object
 7   disclosure_code                   0 non-null      object
 8   annual_avg_estabs                 10 non-null     object
 9   annual_avg_emplvl                 10 non-null     object
 10  total_annual_wages                10 non-null     object
 11  taxable_annual_wages              10 non-null     object
 12  annual_contributions     

## 4. Identify the Analytical Unit of Observation

The QCEW singlefile combines multiple geographic levels, industry levels, and ownership categories in a single dataset. Therefore, the analytical population must be defined explicitly before constructing the Virginia panel.

For this project, the primary unit of analysis will eventually be:

**Virginia locality × industry × year**

Virginia is unusual because independent cities are treated as county-equivalent geographic units. Both counties and independent cities are therefore relevant to the regional analysis.

Before filtering the full dataset, the coding structure for geography, ownership, and industry aggregation is examined using the 2025 source file.

In [14]:
# Inspect the coding structure using only the fields required for record selection.
# -------------------------------------------------------------------------------
# Reading a restricted set of columns reduces memory use while still allowing us
# to determine which geographic, ownership, and industry aggregation levels exist.

selection_columns = [
    "area_fips",
    "own_code",
    "industry_code",
    "agglvl_code",
    "size_code",
    "year"
]

with zipfile.ZipFile(sample_zip_path, "r") as z:
    with z.open(csv_name) as csv_file:
        qcew_structure = pd.read_csv(
            csv_file,
            usecols=selection_columns,
            dtype=str
        )

print(f"Rows loaded: {len(qcew_structure):,}")
print(f"Columns loaded: {qcew_structure.shape[1]}")

Rows loaded: 3,088,320
Columns loaded: 6


In [15]:
# Examine the aggregation levels present in the annual QCEW file.

aggregation_counts = (
    qcew_structure["agglvl_code"]
    .value_counts()
    .sort_index()
    .rename_axis("agglvl_code")
    .reset_index(name="record_count")
)

aggregation_counts

,agglvl_code,record_count
0,10,1
1,11,4
2,12,8
3,13,47
4,14,76
5,15,293
6,16,758
7,17,1409
8,18,1896
9,30,184


In [16]:
# Examine ownership codes represented in the dataset.

ownership_counts = (
    qcew_structure["own_code"]
    .value_counts()
    .sort_index()
    .rename_axis("own_code")
    .reset_index(name="record_count")
)

ownership_counts

,own_code,record_count
0,0,4451
1,1,118969
2,2,149851
3,3,227776
4,5,2587218
5,8,54
6,9,1


In [17]:
# Identify records associated with Virginia.
# -------------------------------------------
# Virginia uses state FIPS code 51. QCEW area identifiers for Virginia counties
# and independent cities therefore begin with "51".

virginia_structure = qcew_structure[
    qcew_structure["area_fips"].str.startswith("51", na=False)
].copy()

print(f"Virginia-related records: {len(virginia_structure):,}")

virginia_structure.head(20)

Virginia-related records: 123,523


,area_fips,own_code,industry_code,agglvl_code,size_code,year
2713979,51000,0,10,50,0,2025
2713980,51000,1,10,51,0,2025
2713981,51000,1,101,52,0,2025
2713982,51000,1,1013,53,0,2025
2713983,51000,1,102,52,0,2025
2713984,51000,1,1021,53,0,2025
2713985,51000,1,1022,53,0,2025
2713986,51000,1,1023,53,0,2025
2713987,51000,1,1024,53,0,2025
2713988,51000,1,1025,53,0,2025


In [18]:
# Determine which aggregation levels occur within Virginia-coded records.

virginia_aggregation_counts = (
    virginia_structure["agglvl_code"]
    .value_counts()
    .sort_index()
    .rename_axis("agglvl_code")
    .reset_index(name="record_count")
)

virginia_aggregation_counts

,agglvl_code,record_count
0,50,1
1,51,4
2,52,8
3,53,39
4,54,61
5,55,187
6,56,477
7,57,919
8,58,1262
9,70,134


In [19]:
# Inspect examples specifically from the county-level aggregation family.
#
# BLS aggregation codes in the 70-series represent county-level records.
# We inspect these records before deciding which industry detail to retain.

virginia_county_examples = (
    virginia_structure[
        virginia_structure["agglvl_code"].str.startswith("7", na=False)
    ]
    .head(30)
)

virginia_county_examples

,area_fips,own_code,industry_code,agglvl_code,size_code,year
2716938,51001,0,10,70,0,2025
2716939,51001,1,10,71,0,2025
2716940,51001,1,102,72,0,2025
2716941,51001,1,1021,73,0,2025
2716942,51001,1,1024,73,0,2025
2716943,51001,1,1026,73,0,2025
2716944,51001,1,1028,73,0,2025
2716945,51001,1,44-45,74,0,2025
2716946,51001,1,457,75,0,2025
2716947,51001,1,4571,76,0,2025


### Interpreting QCEW Aggregation and Ownership Codes

QCEW records are identified using numeric codes that describe both the ownership category and the level of geographic and industry aggregation.

Because these codes determine the economic meaning of each observation, descriptive labels are added before selecting the analytical population. This reduces the risk of combining records that represent different levels of aggregation or ownership.

In [20]:
# Human-readable QCEW ownership labels
# ------------------------------------
# These definitions follow the BLS ownership coding system used in
# NAICS-based QCEW files.

ownership_labels = {
    "0": "Total Covered",
    "1": "Federal Government",
    "2": "State Government",
    "3": "Local Government",
    "5": "Private",
    "8": "Total Government",
    "9": "Total UI Covered (Excludes Federal)"
}

ownership_counts["ownership_title"] = (
    ownership_counts["own_code"]
    .map(ownership_labels)
)

ownership_counts

,own_code,record_count,ownership_title
0,0,4451,Total Covered
1,1,118969,Federal Government
2,2,149851,State Government
3,3,227776,Local Government
4,5,2587218,Private
5,8,54,Total Government
6,9,1,Total UI Covered (Excludes Federal)


In [21]:
# Isolate county-level aggregation codes appearing in Virginia.

county_aggregation_counts = (
    virginia_structure[
        virginia_structure["agglvl_code"].str.startswith("7", na=False)
    ]["agglvl_code"]
    .value_counts()
    .sort_index()
    .rename_axis("agglvl_code")
    .reset_index(name="record_count")
)

county_aggregation_counts

,agglvl_code,record_count
0,70,134
1,71,536
2,72,842
3,73,3048
4,74,4462
5,75,11550
6,76,22828
7,77,35627
8,78,41537


In [22]:
# Examine how aggregation codes map to combinations of
# ownership and industry-code structure.

county_code_examples = (
    virginia_structure[
        virginia_structure["agglvl_code"].str.startswith("7", na=False)
    ]
    .groupby("agglvl_code")
    .first()
    .reset_index()
    [
        [
            "agglvl_code",
            "own_code",
            "industry_code",
            "area_fips",
            "size_code"
        ]
    ]
)

county_code_examples

,agglvl_code,own_code,industry_code,area_fips,size_code
0,70,0,10,51001,0
1,71,1,10,51001,0
2,72,1,102,51001,0
3,73,1,1021,51001,0
4,74,1,44-45,51001,0
5,75,1,457,51001,0
6,76,1,4571,51001,0
7,77,1,45711,51001,0
8,78,1,457110,51001,0


## 5. Define the Primary Industry Analysis Level

For the primary regional industry analysis, the project uses county-level NAICS sector records for private-sector employment.

The selected population is:

- **Geography:** Virginia counties and independent cities
- **Aggregation level:** QCEW level 74 — County × NAICS Sector × Ownership
- **Ownership:** Code 5 — Private sector
- **Industry detail:** Broad NAICS sectors

This level provides an appropriate balance between economic interpretability and industry detail. More granular industry levels may contain sparse or suppressed observations, while broader supersector categories may obscure meaningful differences in local economic structure.

Total covered county records will be retained separately for overall labor-market indicators.

In [23]:
# Select 2025 Virginia private-sector NAICS sector records.
# ---------------------------------------------------------
# Aggregation level 74 represents county-level NAICS sectors
# reported separately by ownership category.
#
# Ownership code 5 represents private-sector employment.

va_private_sectors_2025 = virginia_structure[
    (virginia_structure["agglvl_code"] == "74")
    & (virginia_structure["own_code"] == "5")
].copy()

print(
    f"2025 Virginia private-sector "
    f"county × sector records: {len(va_private_sectors_2025):,}"
)

va_private_sectors_2025.head(20)

2025 Virginia private-sector county × sector records: 2,570


,area_fips,own_code,industry_code,agglvl_code,size_code,year
2717150,51001,5,11,74,0,2025
2717196,51001,5,22,74,0,2025
2717204,51001,5,23,74,0,2025
2717257,51001,5,31-33,74,0,2025
2717336,51001,5,42,74,0,2025
2717396,51001,5,44-45,74,0,2025
2717503,51001,5,48-49,74,0,2025
2717543,51001,5,51,74,0,2025
2717575,51001,5,52,74,0,2025
2717614,51001,5,53,74,0,2025


In [24]:
# Examine the distinct NAICS sector codes present in the selected population.

sector_codes = (
    va_private_sectors_2025["industry_code"]
    .value_counts()
    .sort_index()
    .rename_axis("industry_code")
    .reset_index(name="record_count")
)

sector_codes

,industry_code,record_count
0,11,121
1,21,90
2,22,108
3,23,134
4,31-33,134
5,42,134
6,44-45,134
7,48-49,133
8,51,132
9,52,134


In [25]:
# Count the number of unique Virginia county-equivalent geographic units
# represented in the private-sector industry panel.

n_localities = va_private_sectors_2025["area_fips"].nunique()
n_sectors = va_private_sectors_2025["industry_code"].nunique()

print(f"Unique Virginia localities: {n_localities}")
print(f"Unique NAICS sector codes: {n_sectors}")

Unique Virginia localities: 134
Unique NAICS sector codes: 20


In [26]:
# Examine how many industry sectors are available for each locality.
# Differences may reflect disclosure suppression or absence of activity
# in particular industries.

locality_sector_coverage = (
    va_private_sectors_2025
    .groupby("area_fips")
    .size()
    .describe()
)

locality_sector_coverage

count    134.000000
mean      19.179104
std        1.116188
min       15.000000
25%       19.000000
50%       20.000000
75%       20.000000
max       20.000000
dtype: float64

## 6. Add Geographic and Industry Metadata

The QCEW annual singlefile is optimized for compact programmatic use and does not include descriptive titles for geographic areas or industries.

To make the analytical dataset interpretable while preserving official coding conventions, the QCEW identifiers will be linked to Bureau of Labor Statistics reference data.

Two key lookup tables are required:

1. **Area metadata** — maps `area_fips` identifiers to geographic names.
2. **Industry metadata** — maps `industry_code` identifiers to NAICS industry titles.

Keeping identifiers and descriptive attributes separate also follows a relational data design in which the analytical fact table can be joined to standardized dimension tables as needed.

### Sector Coverage Assessment

The 2025 private-sector panel contains 134 Virginia county-equivalent localities and 20 broad NAICS sectors.

Most localities have published observations for nearly all sectors, with a median of 20 sectors per locality. A small number of locality-sector combinations are absent. These gaps should not automatically be interpreted as zero employment because QCEW publication is subject to disclosure restrictions, particularly for smaller geographic-industry cells.

Sector-level data therefore provide a useful balance between economic detail and publication coverage for the primary regional analysis.

In [27]:
# Identify localities with incomplete sector coverage.
# -----------------------------------------------------
# A locality can have fewer than 20 published sector observations because
# of either an absence of economic activity or QCEW disclosure restrictions.
# At this stage, these missing combinations are flagged rather than imputed.

locality_sector_counts = (
    va_private_sectors_2025
    .groupby("area_fips")
    .size()
    .reset_index(name="published_sector_count")
    .sort_values(
        ["published_sector_count", "area_fips"]
    )
)

incomplete_localities = locality_sector_counts[
    locality_sector_counts["published_sector_count"] < n_sectors
].copy()

print(
    f"Localities with fewer than {n_sectors} "
    f"published sectors: {len(incomplete_localities)}"
)

incomplete_localities

Localities with fewer than 20 published sectors: 62


,area_fips,published_sector_count
112,51678,15
10,51021,16
25,51051,16
44,51091,16
97,51530,16
24,51049,17
100,51570,17
103,51595,17
105,51610,17
108,51640,17


In [28]:
# Summarize the extent of missing locality-sector combinations.

expected_combinations = n_localities * n_sectors
observed_combinations = len(va_private_sectors_2025)
missing_combinations = expected_combinations - observed_combinations

coverage_rate = observed_combinations / expected_combinations

print(f"Expected locality × sector combinations: {expected_combinations:,}")
print(f"Observed combinations:                   {observed_combinations:,}")
print(f"Unobserved combinations:                 {missing_combinations:,}")
print(f"Overall coverage rate:                   {coverage_rate:.1%}")

Expected locality × sector combinations: 2,680
Observed combinations:                   2,570
Unobserved combinations:                 110
Overall coverage rate:                   95.9%


### Coverage Interpretation

Of the 2,680 theoretically possible locality × sector combinations in 2025, 2,570 are published in the QCEW data, producing an overall coverage rate of **95.9%**.

Sixty-two of Virginia's 134 county-equivalent localities have fewer than all 20 broad private-sector industry observations. Missing combinations are retained as missing rather than automatically assigned zero employment because non-publication may reflect either an absence of reportable activity or BLS confidentiality restrictions.

This distinction is important for later calculations of industry concentration and regional growth.

## 7. Add Geographic and Industry Labels

The analytical records currently use coded identifiers for both geography and industry. These identifiers are useful for joins and validation but are not appropriate for interpretation or reporting.

To improve readability, standardized descriptive labels are attached to the broad NAICS sector codes. Geographic names will also be added using an official reference source so that county and independent-city identifiers remain consistent with federal geographic coding.

In [29]:
# Standard NAICS sector labels used in the QCEW sector-level panel.
# -----------------------------------------------------------------
# These labels convert broad industry codes into readable economic
# categories for analysis, visualization, and reporting.

sector_labels = {
    "11": "Agriculture, Forestry, Fishing and Hunting",
    "21": "Mining, Quarrying, and Oil and Gas Extraction",
    "22": "Utilities",
    "23": "Construction",
    "31-33": "Manufacturing",
    "42": "Wholesale Trade",
    "44-45": "Retail Trade",
    "48-49": "Transportation and Warehousing",
    "51": "Information",
    "52": "Finance and Insurance",
    "53": "Real Estate and Rental and Leasing",
    "54": "Professional, Scientific, and Technical Services",
    "55": "Management of Companies and Enterprises",
    "56": "Administrative and Support and Waste Management",
    "61": "Educational Services",
    "62": "Health Care and Social Assistance",
    "71": "Arts, Entertainment, and Recreation",
    "72": "Accommodation and Food Services",
    "81": "Other Services",
    "99": "Unclassified"
}

In [30]:
# Attach readable industry names.

va_private_sectors_2025["industry_title"] = (
    va_private_sectors_2025["industry_code"]
    .map(sector_labels)
)

# Validate that every selected sector received a label.

missing_industry_labels = (
    va_private_sectors_2025["industry_title"]
    .isna()
    .sum()
)

print(f"Records missing an industry title: {missing_industry_labels}")

Records missing an industry title: 0


In [31]:
# Review the industry reference table.

sector_reference = pd.DataFrame(
    sector_labels.items(),
    columns=["industry_code", "industry_title"]
)

sector_reference

,industry_code,industry_title
0,11,"Agriculture, Forestry, Fishing and Hunting"
1,21,"Mining, Quarrying, and Oil and Gas Extraction"
2,22,Utilities
3,23,Construction
4,31-33,Manufacturing
5,42,Wholesale Trade
6,44-45,Retail Trade
7,48-49,Transportation and Warehousing
8,51,Information
9,52,Finance and Insurance


## 8. Add Virginia Locality Metadata

QCEW geographic records are identified using five-digit FIPS codes. For county-level analysis, the first two digits identify the state and the remaining three digits identify the county or county-equivalent.

Virginia's state FIPS code is **51**. Because Virginia's independent cities are treated as county-equivalent geographic units in federal statistical systems, both counties and independent cities are included in the regional analysis.

An official Census Bureau geographic reference file is used to translate these identifiers into readable locality names. Geographic labels are joined to the QCEW records using the five-digit FIPS code rather than being entered manually, reducing the risk of transcription errors and preserving reproducibility.

In [32]:
# Retrieve an official Census Bureau county reference file.
# ---------------------------------------------------------
# The Census Gazetteer file provides standardized county and
# county-equivalent GEOIDs and names for the United States.

CENSUS_COUNTY_URL = (
    "https://www2.census.gov/geo/docs/maps-data/data/gazetteer/"
    "2025_Gazetteer/2025_Gaz_counties_national.zip"
)

census_county_zip = RAW_DIR / "2025_Gaz_counties_national.zip"

if census_county_zip.exists():
    print("Census county reference file already exists — skipping download.")
else:
    response = requests.get(CENSUS_COUNTY_URL, timeout=60)
    response.raise_for_status()

    census_county_zip.write_bytes(response.content)

    print(
        f"Downloaded Census county reference file "
        f"({census_county_zip.stat().st_size / 1_000_000:.2f} MB)"
    )

Downloaded Census county reference file (0.14 MB)


In [33]:
# Inspect the contents of the Census geographic reference archive.

with zipfile.ZipFile(census_county_zip, "r") as z:
    census_zip_contents = z.namelist()

census_zip_contents

['2025_Gaz_counties_national.txt']

In [35]:
# Load the county reference table.
# --------------------------------
# The Census Gazetteer file is pipe-delimited.
# GEOID is retained as a string because it is a geographic identifier,
# not a numeric quantity.

with zipfile.ZipFile(census_county_zip, "r") as z:
    census_file = z.namelist()[0]

    with z.open(census_file) as file:
        county_reference = pd.read_csv(
            file,
            sep="|",
            dtype=str
        )

# Remove any leading or trailing whitespace from column names.
county_reference.columns = county_reference.columns.str.strip()

county_reference.head()

,USPS,GEOID,GEOIDFQ,ANSICODE,NAME,ALAND,AWATER,ALAND_SQMI,AWATER_SQMI,INTPTLAT,INTPTLONG
0,AL,01001,0500000US01001,00161526,Autauga County,1539631460,25677536,594.455,9.914,32.532237,-86.64644
1,AL,01003,0500000US01003,00161527,Baldwin County,4117933903,1132678359,1589.943,437.33,30.659218,-87.746067
2,AL,01005,0500000US01005,00161528,Barbour County,2292160152,50523213,885.008,19.507,31.870253,-85.405103
3,AL,01007,0500000US01007,00161529,Bibb County,1612188713,9572302,622.47,3.696,33.015893,-87.127148
4,AL,01009,0500000US01009,00161530,Blount County,1670296790,14822589,644.905,5.723,33.977358,-86.56644


In [36]:
# Retain Virginia counties and independent cities only.

va_locality_reference = (
    county_reference[
        county_reference["USPS"].str.strip() == "VA"
    ]
    [["GEOID", "NAME"]]
    .rename(
        columns={
            "GEOID": "area_fips",
            "NAME": "locality_name"
        }
    )
    .copy()
)

# Clean potential whitespace inherited from the source file.
va_locality_reference["area_fips"] = (
    va_locality_reference["area_fips"].str.strip()
)

va_locality_reference["locality_name"] = (
    va_locality_reference["locality_name"].str.strip()
)

print(
    f"Virginia county-equivalent geographies "
    f"in Census reference: {len(va_locality_reference):,}"
)

va_locality_reference.head(10)

Virginia county-equivalent geographies in Census reference: 133


,area_fips,locality_name
2822,51001,Accomack County
2823,51003,Albemarle County
2824,51005,Alleghany County
2825,51007,Amelia County
2826,51009,Amherst County
2827,51011,Appomattox County
2828,51013,Arlington County
2829,51015,Augusta County
2830,51017,Bath County
2831,51019,Bedford County


In [37]:
# Compare geographic identifiers between QCEW and Census.

qcew_fips = set(
    va_private_sectors_2025["area_fips"].unique()
)

census_fips = set(
    va_locality_reference["area_fips"].unique()
)

qcew_not_in_census = sorted(qcew_fips - census_fips)
census_not_in_qcew = sorted(census_fips - qcew_fips)

print(f"QCEW FIPS not found in Census: {len(qcew_not_in_census)}")
print(f"Census FIPS not found in QCEW: {len(census_not_in_qcew)}")

QCEW FIPS not found in Census: 1
Census FIPS not found in QCEW: 0


In [38]:
# Inspect geographic identifiers that appear in QCEW but not
# in the 2025 Census county reference file.

qcew_not_in_census

['51999']

In [39]:
# Inspect the unmatched QCEW geography and its available sector records.

unmatched_qcew_geographies = (
    va_private_sectors_2025[
        va_private_sectors_2025["area_fips"].isin(qcew_not_in_census)
    ]
    [
        [
            "area_fips",
            "industry_code",
            "industry_title",
            "agglvl_code",
            "own_code",
            "year"
        ]
    ]
)

unmatched_qcew_geographies

,area_fips,industry_code,industry_title,agglvl_code,own_code,year
2835594,51999,11,"Agriculture, Forestry, Fishing and Hunting",74,5,2025
2835674,51999,21,"Mining, Quarrying, and Oil and Gas Extraction",74,5,2025
2835705,51999,22,Utilities,74,5,2025
2835729,51999,23,Construction,74,5,2025
2835821,51999,31-33,Manufacturing,74,5,2025
2836383,51999,42,Wholesale Trade,74,5,2025
2836542,51999,44-45,Retail Trade,74,5,2025
2836679,51999,48-49,Transportation and Warehousing,74,5,2025
2836798,51999,51,Information,74,5,2025
2836867,51999,52,Finance and Insurance,74,5,2025


### Geographic Reconciliation

The QCEW data contain one Virginia-coded area, `51999`, that does not appear in the Census county reference file.

BLS identifies this code as **"Unknown Or Undefined, Virginia"** rather than a true county or independent city. It represents employment that cannot be assigned to a specific Virginia county-equivalent geography.

Because the purpose of this project is to compare identifiable regional economies, this residual category is excluded from locality-level analysis. The exclusion is documented explicitly rather than treated as a failed geographic match.

In [40]:
# Remove the BLS residual geography that cannot be assigned
# to a specific Virginia county or independent city.
# ---------------------------------------------------------
# QCEW area code 51999 is defined by BLS as
# "Unknown Or Undefined, Virginia."

UNKNOWN_VA_FIPS = "51999"

va_private_sectors_2025 = (
    va_private_sectors_2025[
        va_private_sectors_2025["area_fips"] != UNKNOWN_VA_FIPS
    ]
    .copy()
)

print(
    "Virginia county-equivalent localities retained:",
    va_private_sectors_2025["area_fips"].nunique()
)

Virginia county-equivalent localities retained: 133


In [41]:
# Recheck geographic coverage after excluding the undefined QCEW area.

qcew_fips = set(
    va_private_sectors_2025["area_fips"].unique()
)

census_fips = set(
    va_locality_reference["area_fips"].unique()
)

qcew_not_in_census = sorted(qcew_fips - census_fips)
census_not_in_qcew = sorted(census_fips - qcew_fips)

print(f"QCEW FIPS not found in Census: {len(qcew_not_in_census)}")
print(f"Census FIPS not found in QCEW: {len(census_not_in_qcew)}")

QCEW FIPS not found in Census: 0
Census FIPS not found in QCEW: 0


In [42]:
# Attach readable Census locality names to the QCEW records.
# -----------------------------------------------------------
# Many sector observations belong to each locality, while each
# FIPS code must correspond to exactly one locality name.

va_private_sectors_2025 = (
    va_private_sectors_2025
    .merge(
        va_locality_reference,
        on="area_fips",
        how="left",
        validate="many_to_one"
    )
)

print(
    "Records missing locality names:",
    va_private_sectors_2025["locality_name"].isna().sum()
)

Records missing locality names: 0


In [43]:
va_private_sectors_2025[
    [
        "area_fips",
        "locality_name",
        "industry_code",
        "industry_title",
        "year"
    ]
].head(20)

,area_fips,locality_name,industry_code,industry_title,year
0,51001,Accomack County,11,"Agriculture, Forestry, Fishing and Hunting",2025
1,51001,Accomack County,22,Utilities,2025
2,51001,Accomack County,23,Construction,2025
3,51001,Accomack County,31-33,Manufacturing,2025
4,51001,Accomack County,42,Wholesale Trade,2025
5,51001,Accomack County,44-45,Retail Trade,2025
6,51001,Accomack County,48-49,Transportation and Warehousing,2025
7,51001,Accomack County,51,Information,2025
8,51001,Accomack County,52,Finance and Insurance,2025
9,51001,Accomack County,53,Real Estate and Rental and Leasing,2025


### Revised Sector Coverage After Geographic Reconciliation

After excluding the QCEW residual geography `51999` ("Unknown Or Undefined, Virginia"), the analytical universe contains 133 identifiable Virginia counties and independent cities.

Sector coverage is recalculated using only these valid county-equivalent geographies so that missing-sector statistics reflect the true regional analysis population.

In [44]:
# Recalculate sector coverage using only valid Virginia county-equivalents.

n_localities = va_private_sectors_2025["area_fips"].nunique()
n_sectors = va_private_sectors_2025["industry_code"].nunique()

expected_combinations = n_localities * n_sectors
observed_combinations = len(va_private_sectors_2025)
missing_combinations = expected_combinations - observed_combinations
coverage_rate = observed_combinations / expected_combinations

print(f"Valid Virginia localities:               {n_localities}")
print(f"NAICS sectors:                           {n_sectors}")
print(f"Expected locality × sector combinations: {expected_combinations:,}")
print(f"Observed combinations:                   {observed_combinations:,}")
print(f"Unobserved combinations:                 {missing_combinations:,}")
print(f"Overall coverage rate:                   {coverage_rate:.1%}")

Valid Virginia localities:               133
NAICS sectors:                           20
Expected locality × sector combinations: 2,660
Observed combinations:                   2,550
Unobserved combinations:                 110
Overall coverage rate:                   95.9%


In [45]:
# Reassess sector availability across valid Virginia localities.

locality_sector_counts = (
    va_private_sectors_2025
    .groupby(["area_fips", "locality_name"])
    .size()
    .reset_index(name="published_sector_count")
    .sort_values(
        ["published_sector_count", "locality_name"]
    )
)

print(
    f"Localities with fewer than {n_sectors} published sectors:",
    (locality_sector_counts["published_sector_count"] < n_sectors).sum()
)

locality_sector_counts["published_sector_count"].describe()

Localities with fewer than 20 published sectors: 62


count    133.000000
mean      19.172932
std        1.118110
min       15.000000
25%       19.000000
50%       20.000000
75%       20.000000
max       20.000000
Name: published_sector_count, dtype: float64

## 9. Build the 2019–2025 Virginia Industry Panel

After validating the geographic and industry structure using the 2025 QCEW file, the same selection logic is applied consistently across all years in the study period.

The analytical panel retains:

- Virginia county and independent-city geographies
- Private-sector employment only
- Broad NAICS sector records
- Annual establishment, employment, wage, and pay measures

The resulting dataset will contain repeated observations for each locality-industry combination over time, allowing the analysis of regional growth, industry concentration, wage trends, and structural economic change.

In [46]:
# Variables required for the longitudinal economic panel.
# --------------------------------------------------------
# Identifier fields define the unit of observation, while the remaining
# variables provide the employment, establishment, and wage measures used
# in the economic analysis.

panel_columns = [
    "area_fips",
    "own_code",
    "industry_code",
    "agglvl_code",
    "year",
    "disclosure_code",
    "annual_avg_estabs",
    "annual_avg_emplvl",
    "total_annual_wages",
    "annual_avg_wkly_wage",
    "avg_annual_pay"
]

In [47]:
def extract_va_private_sector_panel(year, zip_path):
    """
    Extract Virginia private-sector NAICS sector records
    from one annual BLS QCEW archive.

    Parameters
    ----------
    year : int
        QCEW calendar year.

    zip_path : pathlib.Path
        Path to the annual QCEW ZIP archive.

    Returns
    -------
    pandas.DataFrame
        Virginia county-equivalent × private-sector × NAICS-sector
        observations for the selected year.
    """

    with zipfile.ZipFile(zip_path, "r") as z:
        csv_name = z.namelist()[0]

        with z.open(csv_name) as csv_file:
            df = pd.read_csv(
                csv_file,
                usecols=panel_columns,
                dtype={
                    "area_fips": str,
                    "own_code": str,
                    "industry_code": str,
                    "agglvl_code": str,
                    "year": int
                }
            )

    # Keep Virginia-coded geographic records.
    df = df[
        df["area_fips"].str.startswith("51", na=False)
    ].copy()

    # Retain county-level NAICS sector records for private employers.
    df = df[
        (df["agglvl_code"] == "74")
        & (df["own_code"] == "5")
    ].copy()

    # Remove the BLS residual geography that cannot be mapped
    # to a specific Virginia county or independent city.
    df = df[
        df["area_fips"] != UNKNOWN_VA_FIPS
    ].copy()

    return df

In [48]:
# Test the extraction logic on 2025 before processing every year.

test_panel_2025 = extract_va_private_sector_panel(
    year=2025,
    zip_path=qcew_raw_files[2025]
)

print(f"Rows extracted: {len(test_panel_2025):,}")
print(f"Unique localities: {test_panel_2025['area_fips'].nunique()}")
print(f"Unique sectors: {test_panel_2025['industry_code'].nunique()}")

test_panel_2025.head()

Rows extracted: 2,550
Unique localities: 133
Unique sectors: 20


,area_fips,own_code,industry_code,agglvl_code,year,disclosure_code,annual_avg_estabs,annual_avg_emplvl,total_annual_wages,annual_avg_wkly_wage,avg_annual_pay
2717150,51001,5,11,74,2025,NaN,33,199,12405541,1199,62339
2717196,51001,5,22,74,2025,N,4,0,0,0,0
2717204,51001,5,23,74,2025,NaN,84,349,20019021,1104,57388
2717257,51001,5,31-33,74,2025,NaN,30,3229,174286182,1038,53977
2717336,51001,5,42,74,2025,NaN,36,167,10504887,1207,62778


In [49]:
# Validate that the reusable extraction function reproduces
# the previously validated 2025 analytical population.

assert len(test_panel_2025) == len(va_private_sectors_2025)

assert (
    test_panel_2025["area_fips"].nunique()
    == va_private_sectors_2025["area_fips"].nunique()
)

assert (
    test_panel_2025["industry_code"].nunique()
    == va_private_sectors_2025["industry_code"].nunique()
)

print("2025 extraction validation passed.")

2025 extraction validation passed.


## 10. Construct the Full 2019–2025 Virginia Industry Panel

The validated extraction procedure is now applied consistently to each annual QCEW archive from 2019 through 2025.

Each annual extract represents:

**Virginia county-equivalent × private-sector NAICS sector × year**

The annual datasets are then vertically combined into a longitudinal panel. Maintaining a consistent selection rule across years is necessary for valid comparisons of employment, wages, establishments, and industry structure over time.

In [50]:
# Apply the validated extraction function to each year.
# ------------------------------------------------------
# Each annual DataFrame is stored temporarily before the
# observations are concatenated into one longitudinal panel.

annual_panels = []

for year in YEARS:
    print(f"Processing {year}...")

    annual_df = extract_va_private_sector_panel(
        year=year,
        zip_path=qcew_raw_files[year]
    )

    annual_panels.append(annual_df)

print("\nAnnual extraction complete.")

Processing 2019...


C:\Users\Chris\AppData\Local\Temp\ipykernel_1496\1774568068.py:25: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


Processing 2020...
Processing 2021...
Processing 2022...
Processing 2023...
Processing 2024...
Processing 2025...

Annual extraction complete.


In [51]:
# Stack the annual extracts into one longitudinal dataset.

va_industry_panel = pd.concat(
    annual_panels,
    ignore_index=True
)

print(f"Total panel rows: {len(va_industry_panel):,}")
print(
    f"Years represented: "
    f"{va_industry_panel['year'].min()}–{va_industry_panel['year'].max()}"
)

va_industry_panel.head()

Total panel rows: 17,804
Years represented: 2019–2025


,area_fips,own_code,industry_code,agglvl_code,year,disclosure_code,annual_avg_estabs,annual_avg_emplvl,total_annual_wages,annual_avg_wkly_wage,avg_annual_pay
0,51001,5,11,74,2019,NaN,21,150,7084971,910,47338
1,51001,5,22,74,2019,N,2,0,0,0,0
2,51001,5,23,74,2019,NaN,90,391,15810968,777,40403
3,51001,5,31-33,74,2019,NaN,25,3285,117564188,688,35794
4,51001,5,42,74,2019,NaN,36,240,10554067,847,44021


In [52]:
# Verify annual observation counts.
# ---------------------------------
# Differences across years can reflect changes in publication coverage,
# industry activity, disclosure status, or classification structure.

annual_record_counts = (
    va_industry_panel
    .groupby("year")
    .size()
    .reset_index(name="record_count")
)

annual_record_counts

,year,record_count
0,2019,2522
1,2020,2521
2,2021,2524
3,2022,2559
4,2023,2572
5,2024,2556
6,2025,2550


### Attach Standardized Geographic and Industry Labels

The longitudinal panel retains coded identifiers as primary keys while attaching readable geographic and industry labels for interpretation.

The Census county reference is joined using the five-digit FIPS identifier, while broad NAICS sector titles are assigned using the standardized sector reference developed earlier.

In [53]:
# Attach locality names using the validated Census reference table.

va_industry_panel = (
    va_industry_panel
    .merge(
        va_locality_reference,
        on="area_fips",
        how="left",
        validate="many_to_one"
    )
)

# Attach broad NAICS sector titles.

va_industry_panel["industry_title"] = (
    va_industry_panel["industry_code"]
    .map(sector_labels)
)

In [54]:
# Confirm that all geographic and industry identifiers were resolved.

print(
    "Records missing locality names:",
    va_industry_panel["locality_name"].isna().sum()
)

print(
    "Records missing industry titles:",
    va_industry_panel["industry_title"].isna().sum()
)

Records missing locality names: 0
Records missing industry titles: 0


## 12. Identify Disclosure-Suppressed Observations

QCEW publication is subject to confidentiality requirements. When an observation cannot be released, BLS marks the record with a disclosure code of `N`.

For these suppressed records, published employment and wage measures may be zero-filled even though economic activity exists. Consequently, these values must not be interpreted as true zeros.

Suppressed observations are therefore identified explicitly and retained as records while their unavailable economic measures are treated as missing for analytical purposes.

In [55]:
# Summarize disclosure status across the longitudinal panel.

disclosure_summary = (
    va_industry_panel["disclosure_code"]
    .fillna("Published")
    .value_counts()
    .rename_axis("disclosure_status")
    .reset_index(name="record_count")
)

disclosure_summary

,disclosure_status,record_count
0,Published,12515
1,N,5289


In [56]:
# Examine whether disclosure suppression changes over time.

suppression_by_year = (
    va_industry_panel
    .assign(
        is_suppressed=
        va_industry_panel["disclosure_code"].eq("N")
    )
    .groupby("year")
    .agg(
        total_records=("is_suppressed", "size"),
        suppressed_records=("is_suppressed", "sum")
    )
    .reset_index()
)

suppression_by_year["suppression_rate"] = (
    suppression_by_year["suppressed_records"]
    / suppression_by_year["total_records"]
)

suppression_by_year

,year,total_records,suppressed_records,suppression_rate
0,2019,2522,677,0.268438
1,2020,2521,690,0.273701
2,2021,2524,695,0.275357
3,2022,2559,702,0.274326
4,2023,2572,738,0.286936
5,2024,2556,806,0.315336
6,2025,2550,981,0.384706


In [57]:
# Convert economic measures to numeric types.
# --------------------------------------------
# Invalid or non-numeric values are converted to missing values
# rather than causing silent coercion problems.

economic_columns = [
    "annual_avg_estabs",
    "annual_avg_emplvl",
    "total_annual_wages",
    "annual_avg_wkly_wage",
    "avg_annual_pay"
]

for column in economic_columns:
    va_industry_panel[column] = pd.to_numeric(
        va_industry_panel[column],
        errors="coerce"
    )

In [58]:
# Replace suppressed economic measures with NaN.
# ------------------------------------------------
# BLS zero-fills unavailable measures in suppressed cells.
# Converting them to NaN prevents those administrative zeros
# from biasing economic calculations.

suppressed_mask = (
    va_industry_panel["disclosure_code"] == "N"
)

suppressed_measure_columns = [
    "annual_avg_emplvl",
    "total_annual_wages",
    "annual_avg_wkly_wage",
    "avg_annual_pay"
]

va_industry_panel.loc[
    suppressed_mask,
    suppressed_measure_columns
] = np.nan

## 13. Longitudinal Panel Validation

Before calculating economic indicators, the combined panel is tested for structural integrity.

Validation focuses on:

- study-period completeness,
- duplicate locality-industry-year observations,
- geographic coverage,
- sector coverage,
- disclosure suppression,
- and missing analytical measures.

These checks are performed before economic calculations so that unexpected structural issues are identified at the data layer rather than discovered later in the analysis.

In [59]:
# Test the expected primary key:
# locality × industry × year.

duplicate_key_count = (
    va_industry_panel
    .duplicated(
        subset=[
            "area_fips",
            "industry_code",
            "year"
        ]
    )
    .sum()
)

print(
    "Duplicate locality × industry × year records:",
    duplicate_key_count
)

Duplicate locality × industry × year records: 0


In [60]:
# Summarize geographic and industry coverage by year.

panel_coverage_by_year = (
    va_industry_panel
    .groupby("year")
    .agg(
        records=("area_fips", "size"),
        localities=("area_fips", "nunique"),
        sectors=("industry_code", "nunique")
    )
    .reset_index()
)

panel_coverage_by_year

,year,records,localities,sectors
0,2019,2522,133,20
1,2020,2521,133,20
2,2021,2524,133,20
3,2022,2559,133,20
4,2023,2572,133,20
5,2024,2556,133,20
6,2025,2550,133,20


In [61]:
# Review the final structure of the analytical panel.

va_industry_panel[
    [
        "year",
        "area_fips",
        "locality_name",
        "industry_code",
        "industry_title",
        "annual_avg_estabs",
        "annual_avg_emplvl",
        "total_annual_wages",
        "annual_avg_wkly_wage",
        "avg_annual_pay",
        "disclosure_code"
    ]
].head(20)

,year,area_fips,locality_name,industry_code,industry_title,annual_avg_estabs,annual_avg_emplvl,total_annual_wages,annual_avg_wkly_wage,avg_annual_pay,disclosure_code
0,2019,51001,Accomack County,11,"Agriculture, Forestry, Fishing and Hunting",21,150.0,7084971.0,910.0,47338.0,NaN
1,2019,51001,Accomack County,22,Utilities,2,NaN,NaN,NaN,NaN,N
2,2019,51001,Accomack County,23,Construction,90,391.0,15810968.0,777.0,40403.0,NaN
3,2019,51001,Accomack County,31-33,Manufacturing,25,3285.0,117564188.0,688.0,35794.0,NaN
4,2019,51001,Accomack County,42,Wholesale Trade,36,240.0,10554067.0,847.0,44021.0,NaN
5,2019,51001,Accomack County,44-45,Retail Trade,147,1300.0,29947809.0,443.0,23040.0,NaN
6,2019,51001,Accomack County,48-49,Transportation and Warehousing,21,NaN,NaN,NaN,NaN,N
7,2019,51001,Accomack County,51,Information,12,81.0,3804494.0,900.0,46777.0,NaN
8,2019,51001,Accomack County,52,Finance and Insurance,27,145.0,7750263.0,1025.0,53297.0,NaN
9,2019,51001,Accomack County,53,Real Estate and Rental and Leasing,44,100.0,3365366.0,650.0,33794.0,NaN


## 14. Acquisition and Panel Construction Summary

The data-acquisition workflow produced a longitudinal QCEW panel covering Virginia's 133 identifiable counties and independent cities from 2019 through 2025.

The final analytical structure is:

**Locality × Private-Sector NAICS Sector × Year**

Key validation results include:

- 133 Virginia county-equivalent geographies represented in every year
- 20 broad NAICS sectors represented in every year
- no duplicate locality-industry-year records
- complete geographic and industry metadata joins
- explicit identification of confidentiality-suppressed observations
- preservation of published establishment counts while unavailable employment and wage measures are treated as missing

Disclosure suppression is economically important for subsequent analysis. The share of sector records with suppressed employment and wage information increases from approximately 27% in 2019 to approximately 38% in 2025. Subsequent growth, concentration, and regional-comparison calculations must therefore distinguish between true economic values and unavailable observations rather than interpreting suppressed values as zeros.

The cleaned longitudinal panel is saved to the processed-data layer for use in the data-quality and economic-analysis notebooks.

In [62]:
# Select and order variables for the processed analytical dataset.
# ----------------------------------------------------------------
# Technical selection fields such as ownership and aggregation codes
# are retained because they document the population definition, while
# readable geographic and industry labels support interpretation.

processed_columns = [
    "year",
    "area_fips",
    "locality_name",
    "industry_code",
    "industry_title",
    "own_code",
    "agglvl_code",
    "disclosure_code",
    "annual_avg_estabs",
    "annual_avg_emplvl",
    "total_annual_wages",
    "annual_avg_wkly_wage",
    "avg_annual_pay"
]

va_industry_panel_processed = (
    va_industry_panel[processed_columns]
    .sort_values(
        ["area_fips", "industry_code", "year"]
    )
    .reset_index(drop=True)
)

va_industry_panel_processed.head()

,year,area_fips,locality_name,industry_code,industry_title,own_code,agglvl_code,disclosure_code,annual_avg_estabs,annual_avg_emplvl,total_annual_wages,annual_avg_wkly_wage,avg_annual_pay
0,2019,51001,Accomack County,11,"Agriculture, Forestry, Fishing and Hunting",5,74,NaN,21,150.0,7084971.0,910.0,47338.0
1,2020,51001,Accomack County,11,"Agriculture, Forestry, Fishing and Hunting",5,74,NaN,22,150.0,8339362.0,1073.0,55782.0
2,2021,51001,Accomack County,11,"Agriculture, Forestry, Fishing and Hunting",5,74,NaN,23,149.0,8779149.0,1136.0,59053.0
3,2022,51001,Accomack County,11,"Agriculture, Forestry, Fishing and Hunting",5,74,N,30,NaN,NaN,NaN,NaN
4,2023,51001,Accomack County,11,"Agriculture, Forestry, Fishing and Hunting",5,74,N,33,NaN,NaN,NaN,NaN


In [63]:
# Final integrity checks before writing the processed dataset.

assert va_industry_panel_processed["year"].min() == 2019
assert va_industry_panel_processed["year"].max() == 2025

assert (
    va_industry_panel_processed["area_fips"].nunique()
    == 133
)

assert (
    va_industry_panel_processed["industry_code"].nunique()
    == 20
)

assert (
    va_industry_panel_processed
    .duplicated(
        subset=[
            "area_fips",
            "industry_code",
            "year"
        ]
    )
    .sum()
    == 0
)

assert (
    va_industry_panel_processed["locality_name"]
    .isna()
    .sum()
    == 0
)

assert (
    va_industry_panel_processed["industry_title"]
    .isna()
    .sum()
    == 0
)

print("Final panel validation passed.")
print(f"Processed records: {len(va_industry_panel_processed):,}")

Final panel validation passed.
Processed records: 17,804


In [64]:
# Save the validated longitudinal panel to the processed-data layer.
# ------------------------------------------------------------------
# CSV is used as an interoperable analytical format that can be read
# by Python, R, Excel, Power BI, and most statistical software.

processed_panel_path = (
    PROCESSED_DIR
    / "va_qcew_private_sector_panel_2019_2025.csv"
)

va_industry_panel_processed.to_csv(
    processed_panel_path,
    index=False
)

print(f"Saved processed panel to:\n{processed_panel_path}")

Saved processed panel to:
C:\Users\Chris\Documents\virginia-regional-economic-intelligence\data\processed\va_qcew_private_sector_panel_2019_2025.csv


In [65]:
# Verify the saved file exists and report its size.

assert processed_panel_path.exists()

file_size_mb = (
    processed_panel_path.stat().st_size
    / 1_000_000
)

print(f"Processed dataset size: {file_size_mb:.2f} MB")

Processed dataset size: 1.62 MB


In [66]:
# Create a compact annual validation summary for documentation.

annual_validation_summary = (
    va_industry_panel_processed
    .assign(
        is_suppressed=lambda x:
            x["disclosure_code"].eq("N")
    )
    .groupby("year")
    .agg(
        records=("area_fips", "size"),
        localities=("area_fips", "nunique"),
        sectors=("industry_code", "nunique"),
        suppressed_records=("is_suppressed", "sum")
    )
    .reset_index()
)

annual_validation_summary["suppression_rate"] = (
    annual_validation_summary["suppressed_records"]
    / annual_validation_summary["records"]
)

validation_summary_path = (
    OUTPUT_TABLES
    / "annual_panel_validation_summary.csv"
)

annual_validation_summary.to_csv(
    validation_summary_path,
    index=False
)

annual_validation_summary

,year,records,localities,sectors,suppressed_records,suppression_rate
0,2019,2522,133,20,677,0.268438
1,2020,2521,133,20,690,0.273701
2,2021,2524,133,20,695,0.275357
3,2022,2559,133,20,702,0.274326
4,2023,2572,133,20,738,0.286936
5,2024,2556,133,20,806,0.315336
6,2025,2550,133,20,981,0.384706
